<cell_type>markdown</cell_type># Feature Engineering - PCA y Feature Selection

**Proyecto:** Sistema de Clasificación de Acciones S&P 500

**Grupo 27** - Universidad de Los Andes

---

## Objetivo

Evaluar si reducción de dimensionalidad mejora el desempeño de los 3 modelos:
1. PCA con 5, 10, 15 componentes
2. Feature Selection (Top 5, 10, 15 features por importancia)
3. Comparar con modelo completo (17 features)

**Modelos a evaluar:** Logistic Regression, Random Forest, XGBoost (los 3 en paralelo)

In [12]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

## 1. Configuración

In [13]:
# Configurar MLflow
experiment_name = "/sp500-feature-engineering"
mlflow.set_experiment(experiment_name)

# Configuración de modelos (solo random_state)
models_config = {
    'Logistic Regression': (LogisticRegression, {'random_state': 42}),
    'Random Forest': (RandomForestClassifier, {'random_state': 42}),
    'XGBoost': (XGBClassifier, {'random_state': 42})
}

print(f"Experimento: {experiment_name}")
print(f"Modelos a evaluar: {list(models_config.keys())}")

Experimento: /sp500-feature-engineering
Modelos a evaluar: ['Logistic Regression', 'Random Forest', 'XGBoost']


## 2. Carga de Datos

In [14]:
# Cargar datasets
train = pd.read_parquet('../../data/processed/ml_ready/train.parquet')
test = pd.read_parquet('../../data/processed/ml_ready/test.parquet')

# Separar features y target
feature_cols = [col for col in train.columns if col not in ['Ticker', 'Date', 'Target']]

X_train = train[feature_cols]
y_train = train['Target']
X_test = test[feature_cols]
y_test = test['Target']

print(f"Features originales: {len(feature_cols)}")

Features originales: 17


In [15]:
# Función helper para entrenar y evaluar
def train_and_evaluate(model_class, params, X_train, y_train, X_test, y_test):
    """
    Entrena modelo y retorna métricas.
    """
    model = model_class(**params)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba)
    }
    
    return metrics, model

## 3. Experimentos con PCA

Evaluar PCA con 5, 10 y 15 componentes principales.

In [16]:
# Normalizar datos (requerido para PCA)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Datos normalizados para PCA")

Datos normalizados para PCA


In [ ]:
# Almacenar resultados
resultados = []

# Probar PCA con diferentes números de componentes
n_components_list = [5, 10, 15]

for modelo_nombre, (modelo_class, params) in models_config.items():
    print(f"\n{modelo_nombre}")
    
    for n_comp in n_components_list:
        
        with mlflow.start_run(run_name=f"{modelo_nombre}_PCA_{n_comp}"):
            
            # Aplicar PCA
            pca = PCA(n_components=n_comp, random_state=42)
            X_train_pca = pca.fit_transform(X_train_scaled)
            X_test_pca = pca.transform(X_test_scaled)
            
            # Varianza explicada
            varianza = pca.explained_variance_ratio_.sum()
            
            # Entrenar y evaluar
            metrics, model = train_and_evaluate(
                modelo_class, params, 
                X_train_pca, y_train, 
                X_test_pca, y_test
            )
            
            # Registrar en MLflow
            mlflow.log_param("modelo", modelo_nombre)
            mlflow.log_param("config", f"PCA-{n_comp}")
            mlflow.log_param("n_components", n_comp)
            mlflow.log_param("varianza_explicada", varianza)
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(metric_name, metric_value)
            
            # Guardar resultados
            resultados.append({
                'Modelo': modelo_nombre,
                'Config': f'PCA-{n_comp}',
                'Features': n_comp,
                'Varianza': f"{varianza:.2%}",
                'Accuracy': metrics['accuracy'],
                'Precision': metrics['precision'],
                'Recall': metrics['recall'],
                'F1-Score': metrics['f1_score'],
                'ROC-AUC': metrics['roc_auc']
            })
            
            print(f"  PCA-{n_comp}: ROC-AUC={metrics['roc_auc']:.4f}, Var={varianza:.2%}")


=== Logistic Regression - Experimentos PCA ===
  PCA-5: ROC-AUC=0.5095, Var=84.91%
  PCA-10: ROC-AUC=0.5034, Var=99.98%
  PCA-15: ROC-AUC=0.5034, Var=100.00%

=== Random Forest - Experimentos PCA ===
  PCA-5: ROC-AUC=0.5040, Var=84.91%
  PCA-10: ROC-AUC=0.5026, Var=99.98%
  PCA-15: ROC-AUC=0.4954, Var=100.00%

=== XGBoost - Experimentos PCA ===
  PCA-5: ROC-AUC=0.5049, Var=84.91%
  PCA-10: ROC-AUC=0.4976, Var=99.98%
  PCA-15: ROC-AUC=0.5068, Var=100.00%


In [18]:
# Obtener feature importance usando Random Forest
rf_temp = RandomForestClassifier(random_state=42)
rf_temp.fit(X_train, y_train)

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_temp.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 features por importancia:")
print(feature_importance.head(10).to_string(index=False))

Top 10 features por importancia:
      feature  importance
Volume_change    0.078676
      Returns    0.078108
       RSI_14    0.072325
Volatility_10    0.072267
    MACD_diff    0.070033
          OBV    0.069678
     BB_width    0.069091
         MACD    0.062674
       ATR_14    0.061833
  MACD_signal    0.061276


## 4. Experimentos con Feature Selection

Feature Selection basado en importancia de Random Forest.

In [ ]:
# Probar Feature Selection con diferentes números de features
n_features_list = [5, 10, 15]

for modelo_nombre, (modelo_class, params) in models_config.items():
    print(f"\n{modelo_nombre}")
    
    for n_feat in n_features_list:
        
        with mlflow.start_run(run_name=f"{modelo_nombre}_Top_{n_feat}"):
            
            # Seleccionar top N features
            top_features = feature_importance.head(n_feat)['feature'].tolist()
            
            X_train_fs = X_train[top_features]
            X_test_fs = X_test[top_features]
            
            # Entrenar y evaluar
            metrics, model = train_and_evaluate(
                modelo_class, params,
                X_train_fs, y_train,
                X_test_fs, y_test
            )
            
            # Registrar en MLflow
            mlflow.log_param("modelo", modelo_nombre)
            mlflow.log_param("config", f"Top-{n_feat}")
            mlflow.log_param("n_features", n_feat)
            mlflow.log_param("selected_features", top_features)
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(metric_name, metric_value)
            
            # Guardar resultados
            resultados.append({
                'Modelo': modelo_nombre,
                'Config': f'Top-{n_feat}',
                'Features': n_feat,
                'Varianza': 'N/A',
                'Accuracy': metrics['accuracy'],
                'Precision': metrics['precision'],
                'Recall': metrics['recall'],
                'F1-Score': metrics['f1_score'],
                'ROC-AUC': metrics['roc_auc']
            })
            
            print(f"  Top-{n_feat}: ROC-AUC={metrics['roc_auc']:.4f}")


=== Logistic Regression - Experimentos Feature Selection ===
  Top-5: ROC-AUC=0.5110
  Top-10: ROC-AUC=0.4837
  Top-15: ROC-AUC=0.4837

=== Random Forest - Experimentos Feature Selection ===
  Top-5: ROC-AUC=0.4953
  Top-10: ROC-AUC=0.5009
  Top-15: ROC-AUC=0.4964

=== XGBoost - Experimentos Feature Selection ===
  Top-5: ROC-AUC=0.4899
  Top-10: ROC-AUC=0.5013
  Top-15: ROC-AUC=0.4964


## 5. Comparación de Resultados

Top 10 mejores configuraciones basados en ROC-AUC

In [23]:
# Consolidar todos los resultados
df_comparacion = pd.DataFrame(resultados)

# Ordenar por ROC-AUC (métrica más robusta)
df_comparacion = df_comparacion.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print("\nComparación de TOP 10 Configuraciones por ROC-AUC:")
print(df_comparacion.head(10).to_string(index=False))


Comparación de TOP 10 Configuraciones por ROC-AUC:
             Modelo Config  Features Varianza  Accuracy  Precision   Recall  F1-Score  ROC-AUC
Logistic Regression  Top-5         5      N/A  0.517488   0.517035 0.998461  0.681281 0.510953
Logistic Regression  PCA-5         5   84.91%  0.515700   0.516931 0.951520  0.669917 0.509459
            XGBoost PCA-15        15  100.00%  0.504571   0.517738 0.595229  0.553786 0.506780
            XGBoost  PCA-5         5   84.91%  0.507353   0.519218 0.623701  0.566684 0.504901
      Random Forest  PCA-5         5   84.91%  0.504571   0.516573 0.635629  0.569950 0.504045
      Random Forest All-17        17     100%  0.514905   0.522443 0.707580  0.601079 0.503962
Logistic Regression PCA-15        15  100.00%  0.518680   0.517796 0.990766  0.680137 0.503361
Logistic Regression PCA-10        10   99.98%  0.518680   0.517796 0.990766  0.680137 0.503352
      Random Forest PCA-10        10   99.98%  0.503577   0.516783 0.598307  0.554565 0.50260

## 6. Resultados finales

In [22]:
# Mejor configuración global
mejor_fila = df_comparacion.iloc[0]
print(f"\nMejor configuración global (por ROC-AUC):")
print(f"  Modelo: {mejor_fila['Modelo']}")
print(f"  Config: {mejor_fila['Config']}")
print(f"  ROC-AUC: {mejor_fila['ROC-AUC']:.4f}")
print(f"  F1-Score: {mejor_fila['F1-Score']:.4f}")

# Mejor por modelo
print("\n\nMejor configuración por modelo (por ROC-AUC):")
for modelo in ['Logistic Regression', 'Random Forest', 'XGBoost']:
    df_modelo = df_comparacion[df_comparacion['Modelo'] == modelo]
    mejor_modelo = df_modelo.iloc[0]
    print(f"  {modelo}: {mejor_modelo['Config']} (ROC-AUC={mejor_modelo['ROC-AUC']:.4f})")


Mejor configuración global (por ROC-AUC):
  Modelo: Logistic Regression
  Config: Top-5
  ROC-AUC: 0.5110
  F1-Score: 0.6813


Mejor configuración por modelo (por ROC-AUC):
  Logistic Regression: Top-5 (ROC-AUC=0.5110)
  Random Forest: PCA-5 (ROC-AUC=0.5040)
  XGBoost: PCA-15 (ROC-AUC=0.5068)


## 7. Análisis y Decisión

**Métrica principal: ROC-AUC** (más robusta que F1-Score para este problema)

### Observaciones por Modelo:

Completar después de ejecutar con los resultados reales.

### Decisión para Notebook 03:

Basándose en los resultados de ROC-AUC, se seleccionarán las mejores combinaciones modelo + configuración de features para optimizar con Optuna.

## Conclusiones

- Se evaluaron 21 configuraciones (3 modelos × 7 configs)
- Todos los experimentos registrados en MLflow
- Resultados ordenados por ROC-AUC (métrica más confiable que F1-Score)

**Próximo paso:** Optimizar con Optuna las mejores combinaciones modelo + configuración de features.